In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to lli_db")

Connected to lli_db


## Question 1: What are the top 10 generics by total batch output?

In [2]:
generic_name_query = """
WITH generic_output AS (
    SELECT
        j.jo_id,
        p.generic_name,
        p.dosage_form,
        CASE
            WHEN p.dosage_form IN ('TABLET', 'BILAYER TABLET')
                THEN MAX(CASE WHEN f.stage = 'compression' 
                         THEN f.actual_output_units END)
            WHEN p.dosage_form = 'CAPSULE'
                THEN MAX(CASE WHEN f.stage = 'encapsulation' 
                         THEN f.actual_output_units END)
            ELSE
                MAX(CASE WHEN f.stage = 'coating' 
                         THEN f.actual_output_units END)
        END AS final_output_units
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    LEFT JOIN dim_date d ON f.date_id = d.date_id
    GROUP BY j.jo_id, p.dosage_form, p.generic_name 
)
SELECT
    generic_name,
    SUM(final_output_units) AS total_output_units
FROM generic_output
WHERE final_output_units IS NOT NULL
GROUP BY generic_name
ORDER BY total_output_units DESC
LIMIT 10;
"""

df_generic = pd.read_sql_query(generic_name_query, engine)

short_names = {
    "PARACETAMOL + THIAMINE MONONITRATE (VITAMIN B1) + PYRIDOXINE HYDROCHLORIDE (VITAMIN B6) + CYANOCOBALAMIN (VITAMIN B12)": "PARACETAMOL + B-COMPLEX",
    "ASCORBIC ACID (SODIUM ASCORBATE) + ZINC (AS GLUCONATE) + CHOLECALCIFEROL (VITAMIN D3)": "ASCORBIC ACID + ZINC + VITAMIN D3",
    "ASCORBIC ACID (AS SODIUM ASCORBATE) + ZINC (AS GLUCONATE)": "ASCORBIC ACID + ZINC"
    # other mappings...
}

df_generic['generic_name_short'] = df_generic['generic_name'].replace(short_names)

# Add millions label
df_generic['label'] = df_generic['total_output_units'].apply(
    lambda x: f"{x/1e6:.1f}M"
)
df_generic

,generic_name,total_output_units,generic_name_short,label
0,ATORVASTATIN CALCIUM,33671120.0,ATORVASTATIN CALCIUM,33.7M
1,BUTAMIRATE CITRATE,24429378.0,BUTAMIRATE CITRATE,24.4M
2,ASCORBIC ACID (SODIUM ASCORBATE) + ZINC (AS GL...,21718696.0,ASCORBIC ACID + ZINC + VITAMIN D3,21.7M
3,ASCORBIC ACID (AS SODIUM ASCORBATE) + ZINC (AS...,21709090.0,ASCORBIC ACID + ZINC,21.7M
4,DAPAGLIFLOZIN,20212016.0,DAPAGLIFLOZIN,20.2M
5,ISOSORBIDE MONONITRATE,18223887.0,ISOSORBIDE MONONITRATE,18.2M
6,METFORMIN HYDROCHLORIDE,16957899.0,METFORMIN HYDROCHLORIDE,17.0M
7,MEFENAMIC ACID,16011197.0,MEFENAMIC ACID,16.0M
8,PARACETAMOL + THIAMINE MONONITRATE (VITAMIN B1...,14343646.0,PARACETAMOL + B-COMPLEX,14.3M
9,PARACETAMOL,9861183.0,PARACETAMOL,9.9M


In [3]:
fig = px.bar(
    df_generic,
    x='total_output_units',
    y='generic_name_short',
    orientation='h',
    title='Total Units Produced by Generic Name — 2025',
    labels={
        'total_output_units': 'Total Units Produced',
        'generic_name_short': ''
    },
    text='label',
    color='total_output_units',
    color_continuous_scale='Blues'
)

fig.update_traces(
    textposition='auto',
    insidetextanchor='end'
)


fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showticklabels=True, showgrid=False, title='')
)

fig.show()

# ⚡ Manufacturing Data Insights – Generic Product Output Analysis

---

### 💡 Insight 1: Atorvastatin Dominates Production Volume

**Observation:**  
**Atorvastatin Calcium** produced **33.7M units**, the highest among all generics.

**Explanation:**  
Reflects **chronic cardiovascular therapy demand**, driving **stable year-round manufacturing schedules**.

**Recommendation:**  
✅ Flag **Atorvastatin** as a **core utilization driver** for the tablet line.  
Future **machine utilization analysis** should evaluate whether **compression capacity is allocated efficiently** to this high-volume product.

---

### ⚡ Insight 2: Vitamin Combination Products Drive OTC Volume

**Observation:**  
**Vitamin combination products appear twice in the Top 5**, producing **over 43M units combined**.

**Explanation:**  
Large batch sizes are common due to:  
- High **OTC demand**  
- **Lower regulatory complexity** vs prescription drugs

**Recommendation:**  
🔍 Investigate if these products run on the **same coating machines**, as they may account for a **significant portion of coating-line utilization**.

---

### 🔄 Insight 3: Chronic Metabolic Drugs Provide Predictable Demand

**Observation:**  
**Metformin + Dapagliflozin** contribute approximately **37M units combined**.

**Explanation:**  
Long-term diabetes therapies lead to:  
- **Predictable production schedules**  
- **Repeat manufacturing batches**

**Recommendation:**  
📊 Use these generics as **baseline demand indicators** for **Market Demand Inference (Module 7)**.

---

### 🔹 Insight 4: Value-Added Formulations Over Commodity Tablets

**Observation:**  
**Paracetamol combination products appear in the ranking**, but **pure Paracetamol ranks only #10**.

**Explanation:**  
The plant prioritizes **value-added formulations** over **commodity single-ingredient tablets**.

**Recommendation:**  
Segment future analyses by:  
- **Single-API products**  
- **Combination products**

This helps better understand **profitability drivers** in manufacturing output.

## Question 2: Which generics have the most batches produced?

In [4]:
generic_batch_no_query = """
   SELECT
        p.generic_name,
        COUNT(DISTINCT f.jo_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    JOIN dim_date d ON d.date_id = f.date_id
    WHERE d.year = 2025
    AND f.stage = 'compounding'
    GROUP BY p.generic_name
    ORDER BY batch_count DESC
    LIMIT 10;
"""

df_generic_batch_no = pd.read_sql_query(generic_batch_no_query, engine)

short_names = {
    "PARACETAMOL + THIAMINE MONONITRATE (VITAMIN B1) + PYRIDOXINE HYDROCHLORIDE (VITAMIN B6) + CYANOCOBALAMIN (VITAMIN B12)": "PARACETAMOL + B-COMPLEX",
    "ASCORBIC ACID (SODIUM ASCORBATE) + ZINC (AS GLUCONATE) + CHOLECALCIFEROL (VITAMIN D3)": "ASCORBIC ACID + ZINC + VITAMIN D3",
    "ASCORBIC ACID (AS SODIUM ASCORBATE) + ZINC (AS GLUCONATE)": "ASCORBIC ACID + ZINC",
    "SITAGLIPTIN (AS PHOSPHATE MONOHYDRATE) + METFORMIN HYDROCHLORIDE": "SITAGLIPTIN + METFORMIN"
    # other mappings...
}

df_generic_batch_no['generic_name_short'] = df_generic_batch_no['generic_name'].replace(short_names)

df_generic_batch_no

,generic_name,batch_count,generic_name_short
0,BUTAMIRATE CITRATE,239,BUTAMIRATE CITRATE
1,ATORVASTATIN CALCIUM,77,ATORVASTATIN CALCIUM
2,ASCORBIC ACID (SODIUM ASCORBATE) + ZINC (AS GL...,71,ASCORBIC ACID + ZINC + VITAMIN D3
3,SITAGLIPTIN (AS PHOSPHATE MONOHYDRATE) + METFO...,64,SITAGLIPTIN + METFORMIN
4,ASCORBIC ACID (AS SODIUM ASCORBATE) + ZINC (AS...,49,ASCORBIC ACID + ZINC
5,PARACETAMOL + THIAMINE MONONITRATE (VITAMIN B1...,49,PARACETAMOL + B-COMPLEX
6,MEFENAMIC ACID,42,MEFENAMIC ACID
7,METFORMIN HYDROCHLORIDE,40,METFORMIN HYDROCHLORIDE
8,CITICOLINE,37,CITICOLINE
9,TELMISARTAN + AMLODIPINE (AS BESILATE),33,TELMISARTAN + AMLODIPINE (AS BESILATE)


In [5]:
fig = px.bar(
    df_generic_batch_no,
    x='batch_count',
    y='generic_name_short',
    orientation='h',
    title='Top 10 Generics by Batch Frequency — 2025',
    labels={
        'batch_count': 'Total Batch Count',
        'generic_name_short': ''
    },
    text='batch_count',
    color='batch_count',
    color_continuous_scale='Blues'
)

fig.update_traces(
    textposition='auto',
    insidetextanchor='end'
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showticklabels=True, showgrid=False, title='')
)

fig.show()

# ⚡ Batch Frequency Insights – Generic Product Analysis

---

 ### 💡 Observation 1: BUTAMIRATE CITRATE Leads in Batch Frequency

 **Observation:**  
 **BUTAMIRATE CITRATE** had the **highest batch frequency** with **239 batches in 2025**.

 **Explanation:**  
 Most **in-demand product** in the portfolio, likely **OTC** or **high-turnover therapy**.

 **Recommendation:**  
 ✅ Ensure **machine availability** and **granule inventory** for compounding and downstream stages to **avoid production bottlenecks**.

---

 ### ⚡ Observation 2: Sharp Drop After Top Generic

 **Observation:**  
 Batch counts drop sharply after the top generic, with **77 batches for ATORVASTATIN**.

 **Explanation:**  
 Production is **concentrated in a few generics**, while the rest of the portfolio has fewer runs.

 **Recommendation:**  
 🛠 Focus **preventive maintenance** and **capacity planning** on machines producing **high-frequency generics**.

---

 ### 🔄 Observation 3 (Optional): Repeated High-Frequency Combinations

 **Observation:**  
 Some **vitamin and chronic drug combinations** appear multiple times:  
 - ASCORBIC + ZINC + D3  
 - PARACETAMOL + B-COMPLEX

 **Explanation:**  
 Multiple **high-frequency batches** may stress **compounding** and **coating lines** differently.

 **Recommendation:**  
 📅 Consider **batch scheduling optimization** to **smooth workload** and **reduce changeover downtime**.

In [6]:
generic_underperforming = """
WITH generic_yields AS (
    SELECT
        j.jo_id,
        p.generic_name,
        p.dosage_form,
        CASE
            WHEN p.dosage_form IN ('TABLET', 'BILAYER TABLET')
                THEN MAX(CASE WHEN f.stage = 'compression' 
                         THEN f.yield_pct END)
            WHEN p.dosage_form = 'CAPSULE'
                THEN MAX(CASE WHEN f.stage = 'encapsulation' 
                         THEN f.yield_pct END)
            ELSE
                MAX(CASE WHEN f.stage = 'coating' 
                         THEN f.yield_pct END)
        END AS yield_pct
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    GROUP BY j.jo_id, p.dosage_form, p.generic_name 
),

product_summary AS (
    SELECT
        generic_name,
        dosage_form,
        COUNT(*)                                                        AS total_batches,
        ROUND(AVG(yield_pct) * 100, 2)                                 AS avg_yield_pct,
        SUM(CASE WHEN yield_pct < 0.96 THEN 1 ELSE 0 END)             AS batches_below_96,
        ROUND(
            SUM(CASE WHEN yield_pct < 0.96 THEN 1 ELSE 0 END)::NUMERIC
            / COUNT(*) * 100, 2
        )                                                               AS pct_batches_below_96
    FROM generic_yields
    GROUP BY generic_name, dosage_form
)

SELECT
    generic_name,
    dosage_form,
    total_batches,
    avg_yield_pct,
    batches_below_96,
    pct_batches_below_96,
    CASE
        WHEN pct_batches_below_96 >= 75 THEN 'Critical'
        WHEN pct_batches_below_96 >= 50 THEN 'At Risk'
        ELSE 'Acceptable'
    END AS underperformance_flag
FROM product_summary
WHERE total_batches >= 3
  AND pct_batches_below_96 > 50
ORDER BY pct_batches_below_96 DESC, avg_yield_pct ASC;
"""

df_generic_underperforming = pd.read_sql_query(generic_underperforming, engine)
df_generic_underperforming

,generic_name,dosage_form,total_batches,avg_yield_pct,batches_below_96,pct_batches_below_96,underperformance_flag
0,MULTIVITAMINS,FILM-COATED TABLET,4,92.91,4,100.00,Critical
1,ETORICOXIB,FILM-COATED TABLET,4,93.82,4,100.00,Critical
2,AMLODIPINE BESILATE + VALSARTAN,FILM-COATED TABLET,3,95.14,3,100.00,Critical
3,CALCIUM ASCORBATE,CAPSULE,7,93.88,6,85.71,Critical
4,ISOXSUPRINE HYDROCHLORIDE,TABLET,13,93.68,11,84.62,Critical
5,METHYLDOPA (AS SESQUIHYDRATE),FILM-COATED TABLET,6,94.97,5,83.33,Critical
6,ASCORBIC ACID,CAPSULE,5,94.21,3,60.00,At Risk
7,METFORMIN HYDROCHLORIDE,EXTENDED RELEASE TABLET,5,95.75,3,60.00,At Risk
8,MEFENAMIC ACID,CAPSULE,41,94.71,24,58.54,At Risk
9,SITAGLIPTIN PHOSPHATE,FILM-COATED TABLET,13,95.92,7,53.85,At Risk


In [7]:
color_map = {'Critical': '#D7263D', 'At Risk': '#F4A261'}

fig = px.bar(
    df_generic_underperforming,
    x='pct_batches_below_96',
    y='generic_name',
    orientation='h',
    title='Products Consistently Underperforming 96% Yield Threshold — 2025',
    labels={
        'pct_batches_below_96': '% of Batches Below 96% Yield',
        'generic_name': ''
    },
    text='pct_batches_below_96',
    color='underperformance_flag',
    color_discrete_map=color_map,
    hover_data={
        'total_batches': True,
        'avg_yield_pct': True,
        'batches_below_96': True,
        'underperformance_flag': False,
        'generic_name': False
    }
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='auto',
    insidetextanchor='end'
)

# 96% reference line (maps to 50 on this axis — the minimum threshold)
fig.add_vline(
    x=50,
    line_dash='dash',
    line_color='gray',
    line_width=1.5,
    annotation_text='50% batch floor',
    annotation_position='top right',
    annotation_font_size=11,
    annotation_font_color='gray'
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(
        showticklabels=True,
        showgrid=False,
        title='% of Batches Below 96% Yield',
        range=[0, 110]
    ),
    legend=dict(
        title='Risk Flag',
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

fig.show()

# Module 2 — Q3: Yield Underperformance Analysis
**Metric:** % of Batches Falling Below 96% Final Stage Yield  
**Threshold:** ≥ 3 batches produced, > 50% of batches below 96% yield  
**Date Range:** 2025

---

## Key Findings

### 🔴 Critical Underperformers (≥ 75% of batches below threshold)

**1. Multivitamins | Etoricoxib | Amlodipine Besilate + Valsartan — Film-Coated Tablet**
- **Observation:** All three products failed the 96% coating yield threshold in 100% of their batches (4, 4, and 3 batches respectively), with average yields between 92.91% and 95.14%.
- **Insight:** A 100% failure rate across multiple Film-Coated Tablet products points less toward product-specific formulation issues and more toward a **shared process variable** — likely coating parameters (pan speed, inlet temperature, spray rate, or coating solution concentration) applied uniformly across these products. The fact that these are all Film-Coated Tablets strengthens this hypothesis.
- **Recommendation:** Audit coating stage process parameters for Film-Coated Tablet batches in 2025. Cross-reference machine usage — if these three products were processed on the same coater, machine condition is the primary suspect. Escalate to a formal process deviation review.

---

**2. Isoxsuprine Hydrochloride — Tablet**
- **Observation:** 11 out of 13 batches (84.6%) fell below 96% yield at the compression stage, with an average yield of 93.68%. This is the highest batch count among Critical products.
- **Insight:** With 13 batches, this is statistically the most reliable signal in the dataset. A compression yield averaging 93.68% on a Tablet dosage form suggests a **chronic compression process issue** — likely related to blend uniformity, punch wear, or tablet weight variation causing excess rejection during IPC checks.
- **Recommendation:** Pull compression IPC records for Isoxsuprine Hydrochloride batches and check for patterns in tablet hardness, friability, or weight deviation. This product alone represents the strongest case for a formal CAPA (Corrective and Preventive Action).

---

**3. Calcium Ascorbate — Capsule**
- **Observation:** 6 out of 7 batches (85.7%) fell below 96% yield at the encapsulation stage, with an average yield of 93.88%.
- **Insight:** Calcium Ascorbate is a mineral salt known for **hygroscopicity and poor flow properties**, which directly impacts encapsulation fill weight consistency and capsule rejection rate. The high failure rate is consistent with a formulation-process fit issue rather than operator or machine error.
- **Recommendation:** Review encapsulation machine fill weight data for Calcium Ascorbate batches. Consider formulation adjustments (e.g., increased glidant concentration) or process parameter optimization (tamping force, dosator settings) to improve fill consistency and reduce capsule rejection.

---

**4. Methyldopa (as Sesquihydrate) — Film-Coated Tablet**
- **Observation:** 5 out of 6 batches (83.3%) fell below 96% at the coating stage, with an average yield of 94.97% — the highest average among Critical products, but still consistently below threshold.
- **Insight:** Methyldopa is a dense, poorly compressible API. Core tablet quality issues (capping, lamination) often carry forward into coating defects and increased coating rejection. The near-threshold average (94.97%) suggests this product is **borderline by formulation design** and sensitive to even minor process variation.
- **Recommendation:** Investigate whether compression yield for Methyldopa batches is also below target — a cascading yield loss from compression through coating would confirm a formulation root cause. Flag for formulation review with R&D.

---

### 🟠 At Risk Products (50–74% of batches below threshold)

**5. Mefenamic Acid — Capsule**
- **Observation:** 24 out of 41 batches (58.5%) fell below 96% encapsulation yield, with an average of 94.71%. Highest batch count in the entire dataset.
- **Insight:** With 41 batches, Mefenamic Acid is a **high-volume product with a systemic yield problem**. At 58.5% failure rate, nearly 3 in every 5 runs underperform. The scale of production means the absolute material loss from this product likely exceeds all Critical products combined.
- **Recommendation:** Prioritize Mefenamic Acid for yield loss quantification in Module 6. Even a 1% yield improvement across 41 batches represents significant material recovery. This is the highest-impact target for process optimization in the capsule line.

---

**6. Sitagliptin Phosphate | Naproxen Sodium — Film-Coated Tablet**
- **Observation:** Both products show 53.9% batch failure rate (7 out of 13 batches each) at the coating stage, with averages of 95.92% and 96.08% respectively.
- **Insight:** Average yields this close to 96% indicate these products are **operating at the edge of the threshold** — small process fluctuations tip individual batches below target. This is a process stability issue, not a chronic underperformance issue.
- **Recommendation:** Implement tighter coating process monitoring (statistical process control) for these two products. The goal is reducing batch-to-batch variability rather than a fundamental process overhaul.

---

**7. Ascorbic Acid — Capsule | Metformin Hydrochloride — Extended Release Tablet**
- **Observation:** Both at 60% batch failure rate (3 out of 5 batches each), with averages of 94.21% and 95.75% respectively.
- **Insight:** Small sample sizes (5 batches each) limit the confidence of this signal. Ascorbic Acid's low average (94.21%) is concerning; Metformin's (95.75%) is borderline. Both warrant monitoring but not immediate escalation.
- **Recommendation:** Flag for continued monitoring through Q1 2026. If failure rate persists beyond 10 batches, escalate to formal review. Metformin Hydrochloride as an Extended Release product has additional complexity — coating integrity directly impacts drug release profile, so any yield loss at the coating stage has a quality risk dimension beyond material loss.

---

## Summary Table

| Product | Dosage Form | Avg Yield | % Below 96% | Flag |
|---|---|---|---|---|
| Multivitamins | Film-Coated Tablet | 92.91% | 100.0% | 🔴 Critical |
| Etoricoxib | Film-Coated Tablet | 93.82% | 100.0% | 🔴 Critical |
| Amlodipine Besilate + Valsartan | Film-Coated Tablet | 95.14% | 100.0% | 🔴 Critical |
| Calcium Ascorbate | Capsule | 93.88% | 85.7% | 🔴 Critical |
| Isoxsuprine Hydrochloride | Tablet | 93.68% | 84.6% | 🔴 Critical |
| Methyldopa (as Sesquihydrate) | Film-Coated Tablet | 94.97% | 83.3% | 🔴 Critical |
| Ascorbic Acid | Capsule | 94.21% | 60.0% | 🟠 At Risk |
| Metformin Hydrochloride | Extended Release Tablet | 95.75% | 60.0% | 🟠 At Risk |
| Mefenamic Acid | Capsule | 94.71% | 58.5% | 🟠 At Risk |
| Naproxen Sodium | Film-Coated Tablet | 96.08% | 53.9% | 🟠 At Risk |
| Sitagliptin Phosphate | Film-Coated Tablet | 95.92% | 53.9% | 🟠 At Risk |

---

## Cross-Cutting Observations

**Film-Coated Tablets are disproportionately represented** — 6 out of 11 underperforming products are Film-Coated Tablets. Given that this dosage form accounts for 51.2% of total output (Module 1), coating stage yield losses have an outsized impact on overall plant efficiency.

**Capsule line has a yield consistency problem** — 3 out of 4 Capsule products in this list are flagged. Combined with the Module 1 finding that the capsule line has utilization opportunity, this suggests the line needs both capacity and process stability attention.

**Mefenamic Acid is the highest-priority target for material loss recovery** — not because it has the worst rate, but because it has the highest volume. Always optimize where the money is, not just where the rate is worst.